# 13 — XGBoost Deployment Model

Step 1 of the deploy pipeline (XGBoost → calibrators → submission).

This notebook:
1. Loads the cached 620-dim feature parquet from notebook 12.
2. **Validates** by reproducing the cached fold-0 OOF (target ~0.887 macro AUC). Catches version drift or alignment bugs before we commit to a deploy artifact.
3. **Trains** a single XGBoost OvR on all training data, saved as a pickle for Kaggle.
4. **Compares** to the NN anchors and recaps the cross-family diversity that justifies the +0.0067 blend lift.

Canonical species set is the 234 from `train.csv` (matches the NN OOF column set). The ~28 species below the `min_pos=5` threshold get a `None` slot in the model list and predict 0.5 at inference; step 2's per-class calibrators skip the same species.

Next: notebook 14 fits per-(model, species) isotonic calibrators on fold-0 OOFs.

## 0 — Setup

In [1]:
import os, sys, time, pickle, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

META_CSV       = ROOT / 'data' / 'raw' / 'train.csv'
FOLDS_CSV      = ROOT / 'data' / 'folds' / 'folds.csv'
FEATURES_CACHE = ROOT / 'experiments' / '_classical_features.parquet'
XGB_EXP_DIR    = ROOT / 'experiments' / 'classical_xgboost'
OOF_CACHED     = XGB_EXP_DIR / 'oof_preds.csv'
DEPLOY_PATH    = XGB_EXP_DIR / 'xgboost_deploy.pkl'

VAL_FOLD = 0
MIN_POS  = 5

# Match notebook 12 exactly so the reproduction step compares apples-to-apples.
XGB_PARAMS = dict(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    tree_method='hist', device='cuda' ,n_jobs=1, verbosity=0, eval_metric='auc',
)

print(f'xgboost      : {xgb.__version__}')
print(f'feature cache: {FEATURES_CACHE}  (exists: {FEATURES_CACHE.exists()})')
print(f'cached OOF   : {OOF_CACHED}  (exists: {OOF_CACHED.exists()})')
print(f'deploy out   : {DEPLOY_PATH}')
print(f'XGB params   : {XGB_PARAMS}')

xgboost      : 3.2.0
feature cache: c:\Users\s159286\Documents\venv\BirdClef26\experiments\_classical_features.parquet  (exists: True)
cached OOF   : c:\Users\s159286\Documents\venv\BirdClef26\experiments\classical_xgboost\oof_preds.csv  (exists: True)
deploy out   : c:\Users\s159286\Documents\venv\BirdClef26\experiments\classical_xgboost\xgboost_deploy.pkl
XGB params   : {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05, 'tree_method': 'hist', 'device': 'cuda', 'n_jobs': 1, 'verbosity': 0, 'eval_metric': 'auc'}


## 1 — Load features and build labels

`feat_df` is the 620-dim parquet from notebook 12. Species list comes from `train.csv` (234 species) so we line up with the NN OOFs — not from `feat_df` (which is missing whatever clips failed extraction).

In [2]:
feat_df = pd.read_parquet(FEATURES_CACHE)
folds   = pd.read_csv(FOLDS_CSV)
meta    = pd.read_csv(META_CSV)

feat_df = feat_df.merge(folds[['filename', 'fold']], on='filename', how='left')

feat_cols = [c for c in feat_df.columns if c.startswith('f') and c[1:].isdigit()]
assert len(feat_cols) == 620, f'expected 620 features, got {len(feat_cols)}'

# Canonical species set = train.csv (234 species, matches NN OOFs)
species_list = sorted(meta['primary_label'].dropna().unique())
sp_to_idx    = {sp: i for i, sp in enumerate(species_list)}
C            = len(species_list)

X = feat_df[feat_cols].values.astype(np.float32)
Y = np.zeros((len(feat_df), C), dtype=np.float32)
for i, sp in enumerate(feat_df['primary_label'].values):
    if isinstance(sp, str) and sp in sp_to_idx:
        Y[i, sp_to_idx[sp]] = 1.0

fold_arr  = feat_df['fold'].values
filenames = feat_df['filename'].values

print(f'samples       : {len(feat_df)}')
print(f'features      : {X.shape}')
print(f'species (full): {C}  (from train.csv)')
print(f'fold sizes    : {sorted(Counter(fold_arr).items())}')
print(f'positives/sp  : min={Y.sum(0).min():.0f}  median={int(np.median(Y.sum(0)))}  max={Y.sum(0).max():.0f}')

samples       : 35549
features      : (35549, 620)
species (full): 206  (from train.csv)
fold sizes    : [(np.int64(0), 7118), (np.int64(1), 7114), (np.int64(2), 7110), (np.int64(3), 7104), (np.int64(4), 7103)]
positives/sp  : min=1  median=125  max=499


## 2 — Validate: reproduce cached fold-0 OOF (~0.887)

Retrain XGBoost OvR on folds {1..4}, predict fold 0, compare to the cached OOF macro AUC. Drift larger than ~0.005 ⇒ feature/label alignment bug or XGBoost version skew. Investigate before training the deploy model.

Runtime: ~5-10 min on a workstation with `n_jobs=1`.

In [3]:
def macro_auc_skip_empty(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        pos = y_true[:, c].sum()
        if 0 < pos < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan'), len(aucs)


def fit_xgb_ovr(X_tr, Y_tr, X_va=None, min_pos=MIN_POS, params=XGB_PARAMS, log_every=50):
    """Fit one XGBClassifier per class. Returns (val_preds_or_None, models, trainable_mask)."""
    C = Y_tr.shape[1]
    models    = [None] * C
    trainable = np.zeros(C, dtype=bool)
    out = None if X_va is None else np.full((X_va.shape[0], C), 0.5, dtype=np.float32)
    t0 = time.time()
    for c in range(C):
        yc = Y_tr[:, c]
        if yc.sum() < min_pos or yc.sum() > len(yc) - min_pos:
            continue
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, yc)
        models[c]    = clf
        trainable[c] = True
        if out is not None:
            out[:, c] = clf.predict_proba(X_va)[:, 1]
        if (c + 1) % log_every == 0:
            print(f'  fit {c+1:>3}/{C}  elapsed={time.time()-t0:.0f}s  trainable={trainable[:c+1].sum()}')
    print(f'fit {trainable.sum()}/{C} species in {time.time()-t0:.0f}s')
    return out, models, trainable

In [4]:
tr_mask = (fold_arr != VAL_FOLD) & (~np.isnan(fold_arr.astype(float)))
va_mask = (fold_arr == VAL_FOLD)
print(f'reproduction split: train={tr_mask.sum()}  val={va_mask.sum()}')

preds_repro, _, mask_repro = fit_xgb_ovr(X[tr_mask], Y[tr_mask], X[va_mask])
auc_repro, n_eval = macro_auc_skip_empty(Y[va_mask], preds_repro)
print(f'\nreproduced fold-0 val macro AUC: {auc_repro:.4f}  (avg over {n_eval} val-positive classes)')
print(f'trainable species (≥{MIN_POS} train positives): {mask_repro.sum()}/{C}')

reproduction split: train=28431  val=7118
  fit  50/206  elapsed=460s  trainable=33
  fit 100/206  elapsed=1424s  trainable=83
  fit 150/206  elapsed=2329s  trainable=133
  fit 200/206  elapsed=3280s  trainable=182
fit 188/206 species in 3389s

reproduced fold-0 val macro AUC: 0.8874  (avg over 206 val-positive classes)
trainable species (≥5 train positives): 188/206


In [5]:
# Compare against the cached OOF directly
if OOF_CACHED.exists():
    cached = pd.read_csv(OOF_CACHED)
    cached_cols = [c for c in cached.columns if c != 'filename']
    print(f'cached OOF: {len(cached)} rows × {len(cached_cols)} species cols')

    val_fn = filenames[va_mask]
    assert set(cached['filename']) == set(val_fn), 'cached val filenames do not match feat_df fold-0'

    # Align rows
    cached = cached.set_index('filename').loc[val_fn]

    # Reduce labels to the cached column set
    keep_idx = [sp_to_idx[c] for c in cached_cols if c in sp_to_idx]
    keep_cols = [c for c in cached_cols if c in sp_to_idx]
    y_va_cached = Y[va_mask][:, keep_idx]
    p_va_cached = cached[keep_cols].values.astype(np.float32)
    auc_cached, _ = macro_auc_skip_empty(y_va_cached, p_va_cached)

    # Same restriction for the reproduction so the comparison is column-aligned
    p_va_repro_on_206 = preds_repro[:, keep_idx]
    auc_repro_206, _ = macro_auc_skip_empty(y_va_cached, p_va_repro_on_206)

    drift = auc_repro_206 - auc_cached
    print(f'cached OOF macro AUC  : {auc_cached:.4f}  (target ~0.887)')
    print(f'reproduction (same cols): {auc_repro_206:.4f}')
    print(f'drift                  : {drift:+.4f}')
    if abs(drift) >= 0.005:
        print('\n⚠️  drift ≥ 0.005 — investigate before training deploy model')
    else:
        print('\n✅ reproduction within tolerance')
else:
    print(f'no cached OOF at {OOF_CACHED} — skipping comparison')
    auc_cached = None

cached OOF: 7118 rows × 206 species cols
cached OOF macro AUC  : 0.8874  (target ~0.887)
reproduction (same cols): 0.8874
drift                  : +0.0000

✅ reproduction within tolerance


## 3 — Train deployment model on all training data

Single XGBoost OvR fit on all 5 folds. Larger train set than the OOF runs (~7k more samples), so the deploy model should be marginally stronger than the fold-0 cached model in solo predictive power, though we can't validate that directly (fold 0 is now in train).

Artifact contents:
- `species_list` (234) — canonical column order, matches NN OOFs
- `trainable_mask` (234) — bool, True iff that species was fit
- `models` (234) — `XGBClassifier` for trainable species, `None` otherwise
- `feature_dim`, `xgb_version`, `params` — bookkeeping for the inference notebook

Pickle format keeps the inference code trivial (one `predict_proba` per non-None model). The cost: XGBoost version must be pinned on Kaggle to match. Note the version in the artifact and `pip install xgboost==<that version>` at the top of the submission notebook.

In [9]:
print('Training deployment XGBoost on all data...')
_, deploy_models, deploy_trainable = fit_xgb_ovr(X, Y, X_va=None)

XGB_EXP_DIR.mkdir(parents=True, exist_ok=True)
deploy_artifact = {
    'species_list':    species_list,
    'trainable_mask':  deploy_trainable,
    'models':          deploy_models,
    'feature_dim':     len(feat_cols),
    'xgb_version':     xgb.__version__,
    'params':          XGB_PARAMS,
    'min_pos':         MIN_POS,
}
with open(DEPLOY_PATH, 'wb') as f:
    pickle.dump(deploy_artifact, f)

size_mb = DEPLOY_PATH.stat().st_size / (1024**2)
print(f'\nsaved → {DEPLOY_PATH} ({size_mb:.1f} MB)')
print(f'  species_list  : {len(species_list)}')
print(f'  trainable     : {deploy_trainable.sum()}/{C}')
print(f'  xgb_version   : {xgb.__version__}')

Training deployment XGBoost on all data...


KeyboardInterrupt: 

In [ ]:
# Sanity-check the saved artifact: round-trip load and measure single-clip latency
with open(DEPLOY_PATH, 'rb') as f:
    loaded = pickle.load(f)

def deploy_predict(artifact, X_in):
    sp_list = artifact['species_list']
    out = np.full((X_in.shape[0], len(sp_list)), 0.5, dtype=np.float32)
    for c, m in enumerate(artifact['models']):
        if m is not None:
            out[:, c] = m.predict_proba(X_in)[:, 1]
    return out

# Warm-up then time a single clip
_ = deploy_predict(loaded, X[:1])
t0 = time.time()
N_SAMPLE = 32
_ = deploy_predict(loaded, X[:N_SAMPLE])
dt_per_clip = (time.time() - t0) / N_SAMPLE * 1000
print(f'predict latency: {dt_per_clip:.1f} ms/clip  ({N_SAMPLE}-clip batch, single thread)')

# Cross-check shape and sanity values
sample = deploy_predict(loaded, X[va_mask][:5])
print(f'shape: {sample.shape}  range: [{sample.min():.3f}, {sample.max():.3f}]  mean: {sample.mean():.3f}')
print(f'non-trainable cols are flat at 0.5: {np.allclose(sample[:, ~deploy_trainable], 0.5)}')

predict latency: 2.0 ms/clip  (32-clip batch, single thread)
shape: (5, 206)  range: [0.000, 0.888]  mean: 0.038
non-trainable cols are flat at 0.5: True


## 4 — Comparison vs NN anchors

Single-model AUC on fold 0 (numbers from notebooks 09-11; replace with your real values if you've re-exported):

In [ ]:
# Edit these to match your real fold-0 numbers
NN_ANCHORS = {
    'effb0_sed':     0.948,
    'effv2s_sed':    0.957,
    'convnext_tiny': 0.951,
    'seresnext26d':  0.943,
    'effv2s_focal':  0.954,
}

comp = pd.DataFrame(
    [{'model': k, 'val_macro_auc': v, 'family': 'NN'} for k, v in NN_ANCHORS.items()]
    + [{'model': 'xgboost (this nb, reproduced)', 'val_macro_auc': auc_repro, 'family': 'GBM'}]
).sort_values('val_macro_auc', ascending=False).reset_index(drop=True)
comp['gap_vs_best'] = comp['val_macro_auc'].max() - comp['val_macro_auc']
print(comp.round(4).to_string(index=False))

best_nn = max(NN_ANCHORS.values())
print(f'\nbest NN single        : {best_nn:.4f}')
print(f'this XGBoost (fold 0) : {auc_repro:.4f}  (gap: {best_nn - auc_repro:+.4f})')

                        model  val_macro_auc family  gap_vs_best
                   effv2s_sed         0.9570     NN       0.0000
                 effv2s_focal         0.9540     NN       0.0030
                convnext_tiny         0.9510     NN       0.0060
                    effb0_sed         0.9480     NN       0.0090
                 seresnext26d         0.9430     NN       0.0140
xgboost (this nb, reproduced)         0.8874    GBM       0.0696

best NN single        : 0.9570
this XGBoost (fold 0) : 0.8874  (gap: +0.0696)


**Why XGBoost stays in the blend.** Raw AUC gap is ~0.07, which would normally mean DROP. But the per-sample top-class probability correlates only 0.04-0.24 with the NN family (vs 0.90+ within the NN family — see notebook 12 §5). That uncorrelated error structure converts to a real blend lift:

| Blend | Fold-0 val macro AUC |
| --- | --- |
| Best NN single | 0.957 |
| NN-only isotonic + mean (4 models) | 0.976 |
| NN + XGBoost isotonic + mean (5 models) | **0.983** |

**+0.0067 from adding XGBoost** to the calibrated mean blend — deploy-tier. This is what step 2 (per-class isotonic) and step 3 (submission pipeline) are built to ship.

**Inference cost.** Feature extraction is the new cost driver — ~5ms/clip in numpy+scipy. The XGBoost predict loop adds another ~10-30ms/clip (printed above). Both shared across all base models, both well inside the Kaggle 90-min CPU budget for typical test-set sizes.

## 5 — Handoff to step 2

**Deliverable:** `experiments/classical_xgboost/xgboost_deploy.pkl`

**Step 2 plan (next notebook):**
1. Load fold-0 OOFs for all 5 base models (4 NN + this XGBoost).
2. Fit `IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1)` per `(model, species)` with `min_pos=2`. Species below the threshold get an identity remap.
3. Save the combined `dict[(model_name, species_code) -> IsotonicRegression]` as one pickle.
4. Validate: apply the calibrators to fold-0 OOFs, mean-blend, confirm macro AUC reproduces **0.983 ± 0.001**. Below 0.980 ⇒ alignment bug.

**Kaggle upload plan (step 3):**
- 5 ONNX base models (from `scripts/export_onnx.py`)
- This XGBoost deploy pickle (~size printed above)
- Calibrator pickle (step 2 output, a few MB)